In [ ]:
# OOD / EXTREME ENGINE-SIZE ROBUSTNESS
# Models:
# DCSR, Standard GP, XGBoost, LightGBM,
# CatBoost, Random Forest, Ridge Regression
import sys
sys.path.append("..")

import os
import time
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

In [19]:
# ---------------------------------------------------------
# 1. Paths and settings
# ---------------------------------------------------------

S_old_path = "../data/processed/S_old_temporal_train.csv"
S_new_path = "../data/processed/S_new_temporal_test.csv"

results_dir = "../results/tables"
predictions_dir = "../results/predictions"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)

TARGET = "co2"

NUMERIC_FEATURES = [
    "engine_size",
    "cylinders"
]

CATEGORICAL_FEATURES = [
    "make",
    "model",
    "vehicle_class",
    "transmission",
    "fuel_type"
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Extreme-region thresholds
LOW_PERCENTILE = 5
HIGH_PERCENTILE = 95

DCSR_MODEL_PATH = "../models/dcsr_model.joblib"
GP_MODEL_PATH = "../models/gp_model.joblib"

dcsr_model = joblib.load(DCSR_MODEL_PATH)
gp_model = joblib.load(GP_MODEL_PATH)

ModuleNotFoundError: No module named 'src'

In [ ]:
# ---------------------------------------------------------
# 2. Load the same temporal train/test datasets
# ---------------------------------------------------------

train_df = pd.read_csv(S_old_path)
test_df = pd.read_csv(S_new_path)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

required_columns = ALL_FEATURES + [TARGET]

missing_train = [
    col for col in required_columns
    if col not in train_df.columns
]

missing_test = [
    col for col in required_columns
    if col not in test_df.columns
]

if missing_train or missing_test:
    raise ValueError(
        f"Missing columns. "
        f"Train: {missing_train}; "
        f"Test: {missing_test}"
    )

In [ ]:
# ============================================================
# 3. PREPARE TRAINING AND TEST DATA
# ============================================================

X_train = train_df[ALL_FEATURES].copy()
X_test = test_df[ALL_FEATURES].copy()

y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

print("\nTarget:", TARGET)
print("Training observations:", len(y_train))
print("Test observations:", len(y_test))

In [ ]:
# ============================================================
# 4. DEFINE OOD / EXTREME ENGINE-SIZE REGION
# ============================================================

# Use TRAINING data only to define thresholds.

train_engine = train_df["engine_size"].replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

test_engine = test_df["engine_size"].replace(
    [np.inf, -np.inf],
    np.nan
).to_numpy()

train_min = train_engine.min()
train_max = train_engine.max()

test_min = np.nanmin(test_engine)
test_max = np.nanmax(test_engine)

print("\n" + "=" * 70)
print("ENGINE-SIZE DISTRIBUTION")
print("=" * 70)

print(f"Training minimum: {train_min:.3f} L")
print(f"Training maximum: {train_max:.3f} L")
print(f"Test minimum:     {test_min:.3f} L")
print(f"Test maximum:     {test_max:.3f} L")

In [ ]:
# ============================================================
# 5. STRICT OOD
# ============================================================

# Strict OOD = test engine size outside the training range.

strict_ood_mask = (
    np.isfinite(test_engine)
    & (
        (test_engine < train_min)
        | (test_engine > train_max)
    )
)

strict_ood_count = int(strict_ood_mask.sum())

print("\nStrict OOD observations:", strict_ood_count)

In [ ]:
# ============================================================
# 6. EXTREME ENGINE-SIZE REGION
# ============================================================

# Calculate thresholds ONLY from S_old training data.

low_threshold = train_engine.quantile(
    LOW_PERCENTILE / 100
)

high_threshold = train_engine.quantile(
    HIGH_PERCENTILE / 100
)

extreme_mask = (
    np.isfinite(test_engine)
    & (
        (test_engine <= low_threshold)
        | (test_engine >= high_threshold)
    )
)

extreme_count = int(extreme_mask.sum())

print("\nExtreme engine-size thresholds")
print(
    f"{LOW_PERCENTILE}th percentile: "
    f"{low_threshold:.3f} L"
)
print(
    f"{HIGH_PERCENTILE}th percentile: "
    f"{high_threshold:.3f} L"
)

print(
    "Extreme-region test observations:",
    extreme_count
)

In [ ]:
# ============================================================
# 7. SELECT ROBUSTNESS EVALUATION SET
# ============================================================

if strict_ood_count > 0:

    # Use genuine strict OOD observations.
    robustness_mask = strict_ood_mask
    evaluation_type = "Strict_OOD"

    print("\nUsing STRICT OOD evaluation.")

else:

    # No observations outside the training range.
    # Therefore use extreme engine-size observations.
    robustness_mask = extreme_mask
    evaluation_type = "Extreme_Engine_Size"

    print("\nNo strict OOD observations found.")
    print("Using training-derived extreme engine-size region.")


if robustness_mask.sum() == 0:

    raise ValueError(
        "No robustness observations were found. "
        "Check engine_size values and the temporal datasets."
    )

print("\nSelected robustness observations:",
      int(robustness_mask.sum()))

print("Evaluation type:", evaluation_type)

In [ ]:
# ============================================================
# 8. PREPROCESSING FOR CONVENTIONAL ML MODELS
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_transformer,
            CATEGORICAL_FEATURES
        )
    ]
)

In [ ]:
# ============================================================
# 9. DEFINE CONVENTIONAL ML MODELS
# ============================================================

conventional_models = {

    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    ),

    "Random_Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=1
    ),

    "Ridge": Ridge(
        alpha=1.0
    )
}

In [ ]:
# ============================================================
# 10. TRAIN CONVENTIONAL MODELS
# ============================================================

predictions = {}
training_times = {}
prediction_times = {}
fitted_models = {}

for model_name, estimator in conventional_models.items():

    print("\n" + "-" * 60)
    print(f"Training {model_name}")
    print("-" * 60)

    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ]
    )

    # Training
    start_time = time.time()

    model_pipeline.fit(
        X_train,
        y_train
    )

    training_times[model_name] = (
        time.time() - start_time
    )

    # Prediction
    start_time = time.time()

    y_pred = model_pipeline.predict(
        X_test
    )

    prediction_times[model_name] = (
        time.time() - start_time
    )

    predictions[model_name] = np.asarray(
        y_pred
    )

    fitted_models[model_name] = model_pipeline

    print(
        f"Training time: "
        f"{training_times[model_name]:.2f} sec"
    )

    print(
        f"Prediction time: "
        f"{prediction_times[model_name]:.2f} sec"
    )

In [ ]:
# ============================================================
# 11. PREPARE INPUTS FOR DCSR AND STANDARD GP
# ============================================================

# DCSR and Standard GP use the numerical predictors only.

symbolic_train = (
    train_df[NUMERIC_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

symbolic_test = (
    test_df[NUMERIC_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

# Training-only median imputation.
symbolic_medians = symbolic_train.median()

X_test_symbolic = (
    symbolic_test
    .fillna(symbolic_medians)
    .to_numpy()
)

In [ ]:
# ============================================================
# 12. PREDICT WITH DCSR
# ============================================================

print("\nPredicting with DCSR...")

start_time = time.time()

predictions["DCSR"] = np.asarray(
    dcsr_model.predict(
        X_test_symbolic
    )
)

prediction_times["DCSR"] = (
    time.time() - start_time
)

training_times["DCSR"] = np.nan

In [ ]:
# ============================================================
# 13. PREDICT WITH STANDARD GP
# ============================================================

print("Predicting with Standard GP...")

start_time = time.time()

predictions["Standard_GP"] = np.asarray(
    gp_model.predict(
        X_test_symbolic
    )
)

prediction_times["Standard_GP"] = (
    time.time() - start_time
)

training_times["Standard_GP"] = np.nan

In [ ]:
# ============================================================
# 14. METRIC FUNCTION
# ============================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    n = len(y_true)

    if n == 0:
        return {
            "N": 0,
            "R2": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan
        }

    r2 = (
        r2_score(y_true, y_pred)
        if n >= 2
        else np.nan
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return {
        "N": n,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    }

In [ ]:
# ============================================================
# 15. EVALUATE ALL MODELS
# ============================================================

results = []
prediction_rows = []

for model_name, y_pred in predictions.items():

    y_pred = np.asarray(y_pred)

    # --------------------------------------------------------
    # Overall temporal test
    # --------------------------------------------------------

    overall_metrics = calculate_metrics(
        y_test,
        y_pred
    )

    results.append({
        "Model": model_name,
        "Evaluation_Set": "Overall_S_new",
        "Evaluation_Type": "Temporal_Test",
        **overall_metrics,
        "Training_Time_sec":
            training_times.get(
                model_name,
                np.nan
            ),
        "Prediction_Time_sec":
            prediction_times.get(
                model_name,
                np.nan
            )
    })

    # --------------------------------------------------------
    # Robustness evaluation
    # --------------------------------------------------------

    robustness_metrics = calculate_metrics(
        y_test[robustness_mask],
        y_pred[robustness_mask]
    )

    results.append({
        "Model": model_name,
        "Evaluation_Set":
            evaluation_type,
        "Evaluation_Type":
            "Robustness",
        **robustness_metrics,
        "Training_Time_sec":
            training_times.get(
                model_name,
                np.nan
            ),
        "Prediction_Time_sec":
            prediction_times.get(
                model_name,
                np.nan
            )
    })

    # --------------------------------------------------------
    # Save predictions and residuals
    # --------------------------------------------------------

    model_prediction_df = pd.DataFrame({

        "Model":
            model_name,

        "engine_size":
            test_engine,

        "Actual_CO2":
            y_test,

        "Predicted_CO2":
            y_pred,

        "Residual":
            y_test - y_pred,

        "Absolute_Error":
            np.abs(
                y_test - y_pred
            ),

        "Strict_OOD":
            strict_ood_mask,

        "Extreme_Engine_Size":
            extreme_mask,

        "Robustness_Set":
            robustness_mask
    })

    prediction_rows.append(
        model_prediction_df
    )

In [ ]:
# ============================================================
# 16. CREATE RESULT TABLES
# ============================================================

results_df = pd.DataFrame(
    results
)

predictions_df = pd.concat(
    prediction_rows,
    ignore_index=True
)


# %%
# ============================================================
# 17. SAVE RESULTS
# ============================================================

results_path = os.path.join(
    results_dir,
    "ood_robustness_all_models.csv"
)

predictions_path = os.path.join(
    predictions_dir,
    "ood_predictions_all_models.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

predictions_df.to_csv(
    predictions_path,
    index=False
)


# %%
# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("OOD / EXTREME ENGINE-SIZE ROBUSTNESS RESULTS")
print("=" * 80)

print(
    results_df.to_string(
        index=False
    )
)

print("\n" + "=" * 80)
print("ROBUSTNESS SUMMARY")
print("=" * 80)

print(
    results_df[
        results_df["Evaluation_Type"]
        == "Robustness"
    ][
        [
            "Model",
            "Evaluation_Set",
            "N",
            "R2",
            "RMSE",
            "MAE"
        ]
    ].to_string(
        index=False
    )
)

print("\nResults saved to:")
print(results_path)

print("\nPredictions saved to:")
print(predictions_path)